In [ ]:
#| default_exp write

## Writing and changing

Cell creation, targeted updates, documentation insertion, and examples.

Writing notebooks safely is harder than appending text to a file. A notebook edit needs to preserve cell ids, clear stale outputs, validate Python when possible, export through nbdev automatically, and avoid overwriting the wrong cell.

`write_nb` inserts cells; `replace_str` replaces literal text across notebook cell sources.

There are two writing modes because notebook edits have two different shapes. Use cell-block writes when adding or replacing structured cells, and use literal replacement only for exact renames across existing cells. Both paths stamp nbskill metadata and export affected notebooks automatically when they have an nbdev export target.


### Production contract

The writing tools are production core. They must preserve stable cell ids where possible, clear stale outputs for changed cells, validate new Python, export nbdev modules after source changes, support dry-run review for broad replacements, and verify structured batch edits by reading the notebook back.


In [ ]:
from contextlib import redirect_stdout
from io import StringIO
from pathlib import Path
from fastcore.nbio import read_nb
from nbskill.read import file_context as _example_file_context
from nbskill.write import write_nb as _example_write_nb
from nbskill.foundation import demo_path, remove_demo_path, write_demo_notebook
from nbskill.write import batch_edit_nb, update_cell, write_nb

In [ ]:
with write_demo_notebook("02_write_example.ipynb") as path:
    _example_write_nb(str(path), chr(10).join([
        "%%markdown",
        "## Result",
        "---",
        "%%code",
        "answer = 42",
    ]), replace=True)
    _example_file_context(str(path))

Wrote 2 cells to nbs/data/02_write_example.ipynb using replace
Cell id=38aae1a1: markdown
## Result


In [ ]:
#| export
import ast
import builtins
import copy
import difflib
import glob
import json
import re
from contextlib import redirect_stdout
from io import StringIO
from pathlib import Path

from fastcore.nbio import mk_cell, new_nb
from fastcore.nbio import read_nb
from fastcore.nbio import write_nb as _write_nb
from fastcore.script import Param

from nbskill.execute import exec_nb, run_notebook_test
from nbskill.review import style_check
from nbskill.foundation import (
    cell_class_names, cell_source, clear_outputs, cli_error,
    cli_return, export_notebook, find_cell_by_id, find_cell_by_text,
    is_exported_code_cell, load_cells_text, one_chapter, parse_cells,
    parse_one_cell, replace_cell, short_call_name, source_hash,
    stamp_notebook_metadata, validate_code_cells,
)
from nbskill.parallel import notebook_locks

### Adding cells

`write_nb` is the broad insertion tool. It can append, insert before or after a stable cell id, replace a whole notebook, or write into a named chapter while preserving notebook structure.

In [ ]:
#| export
def _literal_replacement_mode(old_str, new_str):
    return old_str is not None or new_str is not None


In [ ]:
#| export
def _resolve_notebook_paths(path):
    raw = str(path)
    pth = Path(raw).expanduser()
    if any(char in raw for char in "*?[]"):
        candidates = [Path(item) for item in glob.glob(raw, recursive=True)]
    elif pth.is_dir():
        candidates = list(pth.rglob("*.ipynb"))
    elif pth.is_file():
        candidates = [pth]
    else:
        candidates = []
    paths = sorted({candidate for candidate in candidates if candidate.suffix == ".ipynb" and ".ipynb_checkpoints" not in candidate.parts})
    if not paths: cli_error(f"No notebooks matched {path!r}")
    return paths


In [ ]:
#| export
def _literal_cell_diff(before, after, limit=24):
    lines = list(difflib.unified_diff(
        before.splitlines(), after.splitlines(), fromfile="before", tofile="after", lineterm="", n=2,
    ))
    if len(lines) > limit: lines = [*lines[:limit], "... diff truncated ..."]
    return "\n".join(lines)


In [ ]:
#| export
def _replace_literal_in_notebook(nb, old_str, new_str, validate_code=True, collect_details=False):
    changed_cells, matches, details = 0, 0, []
    for cell in nb.cells:
        before = cell_source(cell)
        count = before.count(old_str)
        if not count: continue
        after = before.replace(old_str, new_str)
        if validate_code and getattr(cell, "cell_type", None) == "code": validate_code_cells([mk_cell(after)])
        if collect_details:
            details.append({
                "cell_id": getattr(cell, "id", ""),
                "cell_type": getattr(cell, "cell_type", ""),
                "matches": count,
                "before_hash": source_hash(before),
                "after_hash": source_hash(after),
                "diff": _literal_cell_diff(before, after),
            })
        cell.source = after
        clear_outputs(cell)
        changed_cells += 1
        matches += count
    if matches: stamp_notebook_metadata(nb)
    return changed_cells, matches, details


In [ ]:
#| export
def _format_literal_replacement_details(changed):
    lines = ["Changed cells:"]
    for nb_path, _, _, details in changed:
        for detail in details:
            lines.append(f"- {nb_path} id={detail['cell_id']} type={detail['cell_type']} matches={detail['matches']}")
            if detail["diff"]:
                lines.extend(f"    {line}" for line in detail["diff"].splitlines())
    return "\n".join(lines)


In [ ]:
#| export
def _write_literal_replacements(path, old_str, new_str, run_test=False, validate_code=True, dry_run=False, show_cells=False):
    if old_str in {None, ""}: cli_error("Pass a non-empty old_str for literal replacements")
    if new_str is None: cli_error("Pass new_str for literal replacements")
    paths = _resolve_notebook_paths(path)
    changed = []
    exported = False
    with notebook_locks(*paths):
        for nb_path in paths:
            nb = read_nb(nb_path)
            cells_changed, matches, details = _replace_literal_in_notebook(
                nb, old_str, new_str, validate_code=validate_code, collect_details=show_cells,
            )
            if not matches: continue
            changed.append((nb_path, cells_changed, matches, details))
            if not dry_run:
                _write_nb(nb, nb_path)
                exported = export_notebook(nb, nb_path) is not None or exported
                if run_test: run_notebook_test(nb_path)
    total_matches = sum(matches for _, _, matches, _ in changed)
    total_cells = sum(cells for _, cells, _, _ in changed)
    if not changed:
        msg = f"No matches for {old_str!r} in {len(paths)} notebook(s)"
    else:
        prefix = "Dry run: would replace" if dry_run else "Replaced"
        msg = f"{prefix} {total_matches} matches in {total_cells} cells across {len(changed)} notebook(s)"
        details = "; ".join(f"{nb_path}: {matches} matches/{cells} cells" for nb_path, cells, matches, _ in changed)
        msg += f" ({details})"
        if exported: msg += " and exported with nbdev"
        if show_cells: msg += f"\n{_format_literal_replacement_details(changed)}"
    print(msg)
    return cli_return([path for path, _, _, _ in changed])


In [ ]:
def write_nb(
    path: str,
    cells: Param("Cell block text", str, opt=False, nargs="?") = "",
    before_id: str | None = None,
    after_id: str | None = None,
    chapter: str | None = None,
    replace: bool = False,
    cell_type: str = "code",
    run_test: bool = False,
    run_style: bool = False,
    style_strict: bool = False,
    validate_code: bool = True,
):
    "Write cells to a notebook."
    if before_id and after_id: cli_error("Use either before_id or after_id, not both")
    if (before_id or after_id) and chapter is not None: cli_error("Use id-based insertion or chapter insertion, not both")
    if (before_id or after_id) and replace: cli_error("Use id-based insertion or replace, not both")
    path = Path(path)
    new_cells = parse_cells(cells, cell_type)
    if validate_code: validate_code_cells(new_cells)
    with notebook_locks(path):
        if replace and chapter is None: nb = new_nb(new_cells)
        else:
            nb = read_nb(path) if path.exists() else new_nb([])
            if chapter is not None:
                span = one_chapter(nb.cells, chapter, create=True)
                if replace:
                    del nb.cells[span["start"] + 1:span["end"]]
                    target = span["start"] + 1
                else: target = span["end"]
            elif before_id or after_id:
                idx, _ = find_cell_by_id(nb.cells, before_id or after_id)
                target = idx if before_id else idx + 1
            else: target = len(nb.cells)
            for offset, cell in enumerate(new_cells): nb.cells.insert(target + offset, cell)
        stamp_notebook_metadata(nb)
        _write_nb(nb, path)
        exported = export_notebook(nb, path) is not None
        msg = f"Wrote {len(nb.cells)} cells to {path}"
        if replace: msg += " using replace"
        if chapter is not None: msg += f" in chapter {chapter!r}"
        if before_id: msg += f" before id={before_id}"
        if after_id: msg += f" after id={after_id}"
        if exported: msg += " and exported with nbdev"
        print(msg)
        if run_test: run_notebook_test(path)
        if run_style:
            print(f"Running chstyle on {path}")
            style_check(path, strict=style_strict)
    return cli_return(path)


def replace_str(
    path: str,
    old_str: str,
    new_str: str = "",
    run_test: bool = False,
    validate_code: bool = True,
    dry_run: bool = False,
    show_cells: bool = False,
):
    "Replace literal text across notebook cell sources."
    return _write_literal_replacements(path, old_str, new_str, run_test=run_test, validate_code=validate_code, dry_run=dry_run, show_cells=show_cells)

In [ ]:
def _exercise_direct_update_cell(path):
    write_nb(str(path), "%%code\nvalue = 1\nvalue = value + 1", replace=True)
    cell = read_nb(path).cells[0]
    assert cell.metadata["nbskill"]["cell_type"] == "code"
    before_text = path.read_text(encoding="utf-8")
    out = StringIO()
    with redirect_stdout(out):
        update_cell(str(path), "value = 2", cell_id=cell.id, line_range="2", dry_run=True)
    preview = out.getvalue()
    assert "Dry run: would update lines 2" in preview
    assert "-value = value + 1" in preview and "+value = 2" in preview
    assert path.read_text(encoding="utf-8") == before_text
    update_cell(str(path), "value = 2", cell_id=cell.id, line_range="2")
    assert read_nb(path).cells[0].source == "value = 1\nvalue = 2"
    assert "source_hash" not in read_nb(path).cells[0].metadata["nbskill"]
    update_cell(str(path), "", cell_id=cell.id, line_range="1")
    assert read_nb(path).cells[0].source == "value = 2"
    multi_def = "def first():\n    return 1\n\n\ndef second():\n    return first() + 1"
    update_cell(str(path), multi_def, cell_id=cell.id)
    assert read_nb(path).cells[0].source == multi_def
    update_cell(str(path), cell_id=cell.id, split_before="def second")
    nb = read_nb(path)
    assert len(nb.cells) == 2
    assert nb.cells[0].id == cell.id
    assert nb.cells[0].source == "def first():\n    return 1"
    assert nb.cells[1].source == "def second():\n    return first() + 1"
    replacement = "%%code\ndef alpha():\n    return 'a'\n---\n%%code\ndef beta():\n    return 'b'"
    update_cell(str(path), replacement, cell_id=nb.cells[1].id, split=True)
    nb = read_nb(path)
    assert len(nb.cells) == 3
    assert nb.cells[1].source == "def alpha():\n    return 'a'"
    expected_beta = "def beta():" + chr(10) + "    return 'b'"
    assert nb.cells[2].source == expected_beta
    escaped = 'payload = "line 1' + chr(92) + 'nline 2"'
    update_cell(str(path), escaped, cell_id=nb.cells[2].id, decode_newlines=False)
    assert read_nb(path).cells[2].source == escaped


In [ ]:
with write_demo_notebook("02_write_update.ipynb") as path:
    _exercise_direct_update_cell(path)
    print("direct update_cell preserved escaped newlines")


In [ ]:
root = demo_path("02_write_literal")
try:
    root.mkdir()
    one = root / "one.ipynb"
    two = root / "two.ipynb"
    write_nb(str(one), "%%code\ndef old_name():\n    return 1\n---\n%%markdown\nold_name docs", replace=True)
    write_nb(str(two), "%%code\nvalue = old_name()", replace=True)
    out = StringIO()
    with redirect_stdout(out):
        write_nb(str(root), old_str="old_name", new_str="new_name", dry_run=True, show_cells=True)
    preview = out.getvalue()
    assert "Dry run: would replace" in preview
    assert "Changed cells:" in preview
    assert "id=" in preview and "matches=" in preview
    assert "-def old_name" in preview and "+def new_name" in preview
    assert "old_name" in read_nb(one).cells[0].source
    write_nb(str(root), old_str="old_name", new_str="new_name")
    assert "new_name" in read_nb(one).cells[0].source
    assert "new_name docs" in read_nb(one).cells[1].source
    assert "new_name" in read_nb(two).cells[0].source
    assert read_nb(two).cells[0].metadata["nbskill"]["cell_type"] == "code"
finally:
    remove_demo_path(root)

Wrote 2 cells to nbs/data/02_write_literal/one.ipynb using replace
Wrote 1 cells to nbs/data/02_write_literal/two.ipynb using replace
Replaced 3 matches in 3 cells across 2 notebook(s) (nbs/data/02_write_literal/one.ipynb: 2 matches/2 cells; nbs/data/02_write_literal/two.ipynb: 1 matches/1 cells)


In [ ]:
#| export
def _save_nb(nb, path):
    with notebook_locks(path):
        stamp_notebook_metadata(nb)
        _write_nb(nb, path)
        export_notebook(nb, path)

### Updating one cell

`update_cell` is the surgical tool. It keeps the original cell id, can replace a whole cell or only a line range, clears stale outputs,.

For whole-cell replacement, pass exactly one notebook cell block: optional `%%code`, `%%markdown`, or `%%raw` marker followed by the cell source. Do not include standalone `---` separators; those mean multiple cells and belong with `write_nb` or `batch_edit_nb`. Use `line_range` or `old_str` when replacing only part of a cell.

`update_cell` keeps CLI newline decoding for shell calls, but MCP callers can bypass it with `decode_newlines=False`. The MCP wrapper also accepts `new_lines`, so agents can send exact multiline source directly instead of writing temporary patch files first.

In [ ]:
#| export
def _parse_line_range(line_range, n_lines):
    if line_range is None: return None
    value = str(line_range).strip()
    if not value: return None
    if ":" in value:
        start_s, end_s = value.split(":", 1)
        start = int(start_s) if start_s else 1
        end = int(end_s) if end_s else n_lines
    else:
        start = end = int(value)
    if start < 1 or end < start or end > n_lines:
        cli_error(f"line_range must be 1-based and within 1:{n_lines}; got {line_range!r}")
    return start - 1, end


In [ ]:
#| export
def _replace_line_range(source, line_range, new):
    lines = source.splitlines()
    start, end = _parse_line_range(line_range, len(lines) or 1)
    replacement = [] if new == "" else new.splitlines()
    return "\n".join([*lines[:start], *replacement, *lines[end:]])


In [ ]:
#| export
def _split_before_index(source, split_before):
    lines = source.splitlines()
    marker = str(split_before or "").strip()
    if not marker: cli_error("split_before must be a non-empty string")
    for idx, line in enumerate(lines):
        if marker in line: return idx
    try:
        for idx, line in enumerate(lines):
            if re.search(marker, line): return idx
    except re.error:
        pass
    cli_error(f"split_before did not match any line: {split_before!r}")


In [ ]:
#| export
def _split_cell_sources_before(source, split_before):
    lines = source.splitlines()
    idx = _split_before_index(source, split_before)
    if idx <= 0 or idx >= len(lines):
        cli_error("split_before must leave non-empty source on both sides")
    return "\n".join(lines[:idx]).strip("\n"), "\n".join(lines[idx:]).strip("\n")


In [ ]:
#| export
def _replace_cell_with_cells(nb, idx, cells):
    old_id = getattr(nb.cells[idx], "id", None)
    if old_id is not None and cells: cells[0].id = old_id
    nb.cells[idx:idx + 1] = cells


In [ ]:
#| export
def update_cell(
    path: str,  # Notebook path
    new: Param("Replacement cell source, replacement text, or line-range replacement", str, opt=False, nargs="?") = "",
    new_file: str | None = None,  # Read replacement text from a UTF-8 file
    decode_newlines: bool = True,  # Decode CLI-style literal \\n escapes in new text
    cell_id: str | None = None,  # Stable notebook cell id to update
    old_str: str | None = None,  # Literal text to replace in the target cell, or used to locate the cell when cell_id is absent
    line_range: str | None = None,  # 1-based inclusive lines to replace, e.g. 3 or 3:5
    cell_type: str = "code",  # Default type for whole-cell replacements without %% marker
    run_test: bool = False,  # Execute the notebook with execnb after writing
    validate_code: bool = True,  # Validate changed Python code before writing
    dry_run: bool = False,  # Show the update plan without writing
):
    "Update one notebook cell: replace whole cell, a text substring, or a line range."
    if cell_id is None and old_str is None: cli_error("Pass --cell_id, --old_str, or both")
    if line_range is not None and cell_id is None: cli_error("Pass --cell_id with --line_range")
    path = Path(path)
    new = load_cells_text(new, new_file, decode_newlines=decode_newlines)

    with notebook_locks(path):
        nb = read_nb(path)
        idx, cell = find_cell_by_id(nb.cells, cell_id) if cell_id else find_cell_by_text(nb.cells, old_str)
        if old_str is not None and old_str not in cell_source(cell): cli_error(f"old_str was not found in id={cell.id}")

        if line_range is not None:
            replacement = _replace_line_range(cell_source(cell), line_range, new)
            if validate_code and getattr(cell, "cell_type", None) == "code": validate_code_cells([mk_cell(replacement)])
            mode = f"lines {line_range}"
            if not dry_run:
                cell.source = replacement
                clear_outputs(cell)
        elif old_str is None:
            new_cell = parse_one_cell(new, cell_type)
            if validate_code: validate_code_cells([new_cell])
            clear_outputs(new_cell)
            if not dry_run: replace_cell(nb, idx, new_cell)
            replacement = cell_source(new_cell)
            mode = "cell"
        else:
            replacement = cell_source(cell).replace(old_str, new, 1)
            if validate_code and getattr(cell, "cell_type", None) == "code": validate_code_cells([mk_cell(replacement)])
            mode = "text"
            if not dry_run:
                cell.source = replacement
                clear_outputs(cell)

        msg = f"{'Dry run: would update' if dry_run else 'Updated'} {mode} id={cell.id}"
        if dry_run:
            diff = _literal_cell_diff(cell_source(cell), replacement)
            if diff: msg += chr(10) + diff
            print(msg)
            return cli_return(path)
        stamp_notebook_metadata(nb)
        _write_nb(nb, path)
        if export_notebook(nb, path) is not None: msg += " and exported with nbdev"
        print(msg)
        if run_test: run_notebook_test(path)
    return cli_return(path)


def split_cell(
    path: str,  # Notebook path
    cell_id: str,  # Stable cell id to split
    new: Param("Multi-cell replacement text separated by ---", str, opt=False, nargs="?") = "",
    new_file: str | None = None,  # Read replacement text from a UTF-8 file
    decode_newlines: bool = True,  # Decode CLI-style literal \\n escapes in new text
    split_before: str | None = None,  # Split existing cell before the first line containing or matching this text
    cell_type: str = "code",  # Default type for replacement cells without %% marker
    run_test: bool = False,  # Execute the notebook with execnb after writing
    validate_code: bool = True,  # Validate changed Python code before writing
    dry_run: bool = False,  # Show the split plan without writing
):
    "Split one notebook cell into multiple cells, either at a matching line or by supplying replacement content."
    if split_before is not None and new: cli_error("Pass split_before or new, not both")
    if split_before is None and not new and not new_file: cli_error("Pass split_before or new")
    path = Path(path)
    new = load_cells_text(new, new_file, decode_newlines=decode_newlines)

    with notebook_locks(path):
        nb = read_nb(path)
        idx, cell = find_cell_by_id(nb.cells, cell_id)
        if split_before is not None:
            first, second = _split_cell_sources_before(cell_source(cell), split_before)
            new_cells = [mk_cell(first, cell_type=cell.cell_type), mk_cell(second, cell_type=cell.cell_type)]
            mode = f"split before {split_before!r}"
        else:
            new_cells = parse_cells(new, cell_type)
            if len(new_cells) <= 1: cli_error("new must contain multiple cells separated by ---")
            mode = f"split into {len(new_cells)} cells"
        if validate_code: validate_code_cells(new_cells)
        replacement = "\n---\n".join(cell_source(item) for item in new_cells)
        msg = f"{'Dry run: would split' if dry_run else 'Split'} {mode} id={cell.id}"
        if dry_run:
            diff = _literal_cell_diff(cell_source(cell), replacement)
            if diff: msg += chr(10) + diff
            print(msg)
            return cli_return(path)
        for item in new_cells: clear_outputs(item)
        _replace_cell_with_cells(nb, idx, new_cells)
        stamp_notebook_metadata(nb)
        _write_nb(nb, path)
        if export_notebook(nb, path) is not None: msg += " and exported with nbdev"
        print(msg)
        if run_test: run_notebook_test(path)
    return cli_return(path)


### Structured cell edits

MCP tools and other structured callers send lists of source lines instead of shell-escaped text. These helpers keep that transport shape in the write layer, including optional `split_lines` markers that only split on empty lines.

In [ ]:
#| export
def join_source_lines(lines, field="source_lines"):
    "Return newline text from a structured list of source lines."
    if lines is None: raise ValueError(f"Pass {field} as a list of source lines")
    if not isinstance(lines, list): raise ValueError(f"{field} must be a list of strings")
    return chr(10).join(str(line) for line in lines)

In [ ]:
#| export
def valid_split_line_numbers(source_lines, split_lines=None):
    "Return requested 1-based split lines that point at empty source lines."
    lines = list(source_lines or [])
    valid = []
    for value in split_lines or []:
        try: line_no = int(value)
        except (TypeError, ValueError): continue
        idx = line_no - 1
        if 0 <= idx < len(lines) and str(lines[idx]).strip() == "": valid.append(line_no)
    return sorted(set(valid))

In [ ]:
#| export
def split_source_lines(source_lines, split_lines=None):
    "Split source lines at requested empty 1-based line numbers."
    lines = list(source_lines or [])
    split_at = valid_split_line_numbers(lines, split_lines)
    if not split_at: return [lines]
    chunks, start = [], 0
    for line_no in split_at:
        idx = line_no - 1
        if lines[start:idx]: chunks.append(lines[start:idx])
        start = idx + 1
    if lines[start:]: chunks.append(lines[start:])
    return chunks or [[]]

In [ ]:
#| export
def source_lines_cell(source_lines, cell_type="code"):
    "Build one clean notebook cell from structured source lines."
    cell = mk_cell(join_source_lines(source_lines), cell_type=cell_type)
    clear_outputs(cell)
    return cell

In [ ]:
#| export
def source_lines_cells(cell, default_cell_type="code"):
    "Build one or more cells from a structured cell object."
    cell = cell or {}
    cell_type = cell.get("cell_type", default_cell_type)
    chunks = split_source_lines(cell.get("source_lines", []), cell.get("split_lines"))
    return [source_lines_cell(chunk, cell_type=cell_type) for chunk in chunks]

In [ ]:
#| export
_FEEDBACK_OUTPUT_CALLS = {"display", "print"}

In [ ]:
#| export
def _statement_has_output_call(node):
    if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)): return False
    return any(isinstance(child, ast.Call) and short_call_name(child.func, default='') in _FEEDBACK_OUTPUT_CALLS for child in ast.walk(node))

In [ ]:
#| export
def _source_has_output_feedback(source):
    stripped = str(source or "").strip()
    if not stripped: return False
    if any(line.lstrip().startswith(("%", "!")) for line in stripped.splitlines()): return True
    try: tree = ast.parse(source)
    except SyntaxError: return True
    if any(_statement_has_output_call(node) for node in tree.body): return True
    if not tree.body or not isinstance(tree.body[-1], ast.Expr): return False
    value = tree.body[-1].value
    if isinstance(value, ast.Constant) and isinstance(value.value, str): return False
    return True

In [ ]:
#| export
def should_run_cell_feedback(cell):
    "Return whether an edited cell should be executed for immediate feedback."
    if getattr(cell, "cell_type", None) != "code": return False
    if is_exported_code_cell(cell): return False
    classes = set(cell_class_names(cell))
    if classes & {"example_cell", "test_cell"}: return True
    return _source_has_output_feedback(cell_source(cell))

In [ ]:
#| export
def notebook_edit_feedback(path, cell_ids, auto_feedback=True, timeout=10, safe=True):
    "Execute through the last feedback-worthy edited cell and return visible output."
    if not auto_feedback: return ""
    wanted = {str(cell_id) for cell_id in (cell_ids or []) if cell_id}
    if not wanted: return ""
    nb = read_nb(path)
    targets = [cell for cell in nb.cells if getattr(cell, "id", None) in wanted and should_run_cell_feedback(cell)]
    if not targets: return ""
    target_id = targets[-1].id
    out = StringIO()
    with redirect_stdout(out):
        exec_nb(str(path), up2id=target_id, timeout=timeout, show_output=True, safe=safe, allow_new=True, check_only=True)
    text = out.getvalue().strip()
    return f"Auto feedback (up to id={target_id}):\n{text}" if text else ""

In [ ]:
#| export
def append_notebook_edit_feedback(message, path, cell_ids, auto_feedback=True, feedback_timeout=10, feedback_safe=True):
    "Append automatic notebook execution feedback when an edited cell calls for it."
    feedback = notebook_edit_feedback(path, cell_ids, auto_feedback=auto_feedback, timeout=feedback_timeout, safe=feedback_safe)
    return f"{message}\n\n{feedback}" if feedback else message

In [ ]:
#| export
def save_notebook_edit(nb, path):
    "Stamp, write, export, and report whether nbdev produced Python."
    stamp_notebook_metadata(nb)
    _write_nb(nb, path)
    return export_notebook(nb, path) is not None

In [ ]:
#| export
def replace_notebook_cell(
    path, cell_id, source_lines, cell_type="code", split_lines=None, validate_code=True,
    auto_feedback=True, feedback_timeout=10, feedback_safe=True,
):
    "Replace one notebook cell from structured source lines, optionally splitting on empty lines."
    path = Path(path)
    cell_data = dict(source_lines=source_lines, cell_type=cell_type, split_lines=split_lines)
    new_cells = source_lines_cells(cell_data, default_cell_type=cell_type)
    if validate_code: validate_code_cells(new_cells)
    with notebook_locks(path):
        nb = read_nb(path)
        idx, old_cell = find_cell_by_id(nb.cells, cell_id)
        _replace_cell_with_cells(nb, idx, new_cells)
        new_cell_ids = [cell.id for cell in new_cells]
        exported = save_notebook_edit(nb, path)
    mode = "cell" if len(new_cells) == 1 else f"cell into {len(new_cells)} cells"
    msg = f"Updated {mode} id={getattr(old_cell, 'id', cell_id)}"
    msg += " and exported with nbdev" if exported else ""
    return append_notebook_edit_feedback(msg, path, new_cell_ids, auto_feedback, feedback_timeout, feedback_safe)

In [ ]:
#| export
def replace_notebook_range(
    path, cell_id, start_line, end_line, replacement_lines, validate_code=True,
    auto_feedback=True, feedback_timeout=10, feedback_safe=True,
):
    "Replace one inclusive 1-based line range from structured replacement lines."
    path = Path(path)
    replacement = join_source_lines(replacement_lines, field="replacement_lines")
    with notebook_locks(path):
        nb = read_nb(path)
        _, cell = find_cell_by_id(nb.cells, cell_id)
        cell.source = _replace_line_range(cell_source(cell), f"{start_line}:{end_line}", replacement)
        if validate_code and getattr(cell, "cell_type", None) == "code": validate_code_cells([mk_cell(cell_source(cell))])
        clear_outputs(cell)
        exported = save_notebook_edit(nb, path)
    msg = f"Updated lines {start_line}:{end_line} id={cell_id}"
    msg += " and exported with nbdev" if exported else ""
    return append_notebook_edit_feedback(msg, path, [cell_id], auto_feedback, feedback_timeout, feedback_safe)

In [ ]:
#| export
def insert_notebook_cells(
    path, anchor_id, where="after", cells=None, validate_code=True, default_cell_type="code",
    auto_feedback=True, feedback_timeout=10, feedback_safe=True,
):
    "Insert structured cells before or after an anchor cell id."
    path = Path(path)
    if where not in {"before", "after"}: raise ValueError("where must be 'before' or 'after'")
    new_cells = [item for cell in (cells or []) for item in source_lines_cells(cell, default_cell_type)]
    if not new_cells: raise ValueError("Pass at least one cell")
    if validate_code: validate_code_cells(new_cells)
    with notebook_locks(path):
        nb = read_nb(path)
        idx, _ = find_cell_by_id(nb.cells, anchor_id)
        target = idx if where == "before" else idx + 1
        nb.cells[target:target] = new_cells
        new_cell_ids = [cell.id for cell in new_cells]
        exported = save_notebook_edit(nb, path)
    msg = f"Inserted {len(new_cells)} cell{'s' if len(new_cells) != 1 else ''} {where} id={anchor_id}"
    msg += " and exported with nbdev" if exported else ""
    return append_notebook_edit_feedback(msg, path, new_cell_ids, auto_feedback, feedback_timeout, feedback_safe)

In [ ]:
#| export
def delete_notebook_cell(path, cell_id):
    "Delete one notebook cell by stable id."
    path = Path(path)
    with notebook_locks(path):
        nb = read_nb(path)
        idx, cell = find_cell_by_id(nb.cells, cell_id)
        del nb.cells[idx]
        exported = save_notebook_edit(nb, path)
    msg = f"Deleted cell id={getattr(cell, 'id', cell_id)}"
    return msg + (" and exported with nbdev" if exported else "")

In [ ]:
#| export
def apply_notebook_edit(
    edit, default_path=None, validate_code=True, default_cell_type="code",
    auto_feedback=True, feedback_timeout=10, feedback_safe=True,
):
    "Apply one structured notebook edit operation."
    op = edit.get("op")
    path = str(edit.get("path") or default_path or "")
    if not path: raise ValueError(f"Edit operation missing path: {edit}")
    cell_id = edit.get("cell_id") or edit.get("anchor_id")
    if not cell_id and op not in {"insert_before", "insert_after"}: raise ValueError(f"Edit operation missing cell_id: {edit}")
    if op == "replace_cell":
        return replace_notebook_cell(path, cell_id, edit.get("source_lines"), edit.get("cell_type", default_cell_type), edit.get("split_lines"), validate_code, auto_feedback, feedback_timeout, feedback_safe)
    if op == "replace_range":
        start, end = edit.get("start_line"), edit.get("end_line")
        if start is None or end is None: raise ValueError(f"replace_range needs start_line and end_line: {edit}")
        lines = edit.get("replacement_lines", edit.get("source_lines"))
        return replace_notebook_range(path, cell_id, start, end, lines, validate_code, auto_feedback, feedback_timeout, feedback_safe)
    if op in {"insert_before", "insert_after"}:
        anchor_id = edit.get("anchor_id") or cell_id
        where = "before" if op == "insert_before" else "after"
        return insert_notebook_cells(path, anchor_id, where, edit.get("cells") or [edit], validate_code, default_cell_type, auto_feedback, feedback_timeout, feedback_safe)
    if op == "delete_cell": return delete_notebook_cell(path, cell_id)
    raise ValueError(f"Unknown edit operation {op!r}")

In [ ]:
lines = ["def first():", "    return 1", "", "def second():", "    return 2"]
assert join_source_lines(["a", "b"]) == "a\nb"
assert valid_split_line_numbers(lines, [2, 3]) == [3]
assert split_source_lines(lines, [2, 3]) == [lines[:2], lines[3:]]
assert source_lines_cell(["scratch = True"]).source == "scratch = True"
assert len(source_lines_cells({"source_lines": lines, "split_lines": [3]})) == 2
print("structured line helpers:", len(split_source_lines(lines, [3])))

structured line helpers: 2


In [ ]:
with write_demo_notebook("02_write_structured_edit.ipynb") as path:
    write_nb(str(path), "%%code\nvalue = 1", replace=True)
    cell_id = read_nb(path).cells[0].id
    source = ["def first():", "    return 1", "", "def second():", "    return 2"]
    msg = replace_notebook_cell(str(path), cell_id, source, split_lines=[2, 3])
    nb = read_nb(path)
    assert len(nb.cells) == 2
    range_msg = replace_notebook_range(str(path), nb.cells[0].id, 1, 1, ["def uno():"])
    insert_msg = insert_notebook_cells(str(path), nb.cells[1].id, cells=[{"source_lines": ["tail = True"]}])
    assert read_nb(path).cells[2].source == "tail = True"
    print(msg)
    print(range_msg)
    print(insert_msg)

Wrote 1 cells to nbs/data/02_write_structured_edit.ipynb using replace
Updated cell into 2 cells id=aa02cf6a
Updated lines 1:1 id=aa02cf6a
Inserted 1 cell after id=e05b6833


In [ ]:
with write_demo_notebook("02_write_structured_apply.ipynb") as path:
    write_nb(str(path), "%%code\ntail = True", replace=True)
    tail_id = read_nb(path).cells[0].id
    op = dict(op="replace_cell", cell_id=tail_id, source_lines=["tail = 'applied'"])
    apply_msg = apply_notebook_edit(op, str(path))
    assert read_nb(path).cells[0].source == "tail = 'applied'"
    delete_msg = delete_notebook_cell(str(path), tail_id)
    assert len(read_nb(path).cells) == 0
    assert save_notebook_edit(read_nb(path), path) is False
    print(apply_msg)
    print(delete_msg)
    print("structured helpers left cells:", len(read_nb(path).cells))

Wrote 1 cells to nbs/data/02_write_structured_apply.ipynb using replace
Updated cell id=892ad2a6
Deleted cell id=892ad2a6
structured helpers left cells: 0


In [ ]:
with write_demo_notebook("02_write_feedback.ipynb") as path:
    write_nb(str(path), "%%code\nseed = 1", replace=True)
    cell_id = read_nb(path).cells[0].id
    msg = replace_notebook_cell(str(path), cell_id, ["print('feedback', 3)"])
    assert "Auto feedback" in msg
    assert "feedback 3" in msg
    assert read_nb(path).cells[0].outputs == []
    feedback_cell = read_nb(path).cells[0]
    assert should_run_cell_feedback(feedback_cell)
    assert "feedback 3" in notebook_edit_feedback(str(path), [cell_id])
    assert "feedback 3" in append_notebook_edit_feedback("prefix", str(path), [cell_id])
    quiet = replace_notebook_cell(str(path), cell_id, ["seed = 4"])
    assert "Auto feedback" not in quiet
    print("automatic edit feedback returned output")

Wrote 1 cells to nbs/data/02_write_feedback.ipynb using replace
automatic edit feedback returned output


Batch editing

Agents often need to change several notebook cells together. `batch_edit_nb` accepts a JSON edit plan, runs the same notebook-aware validation and locking as the single-cell tools, and prints a compact diff before writing. Use it when repeated shell calls would make multiline code or dry-run diffs fragile.

In [ ]:
#| export
def _load_batch_plan(plan="", plan_file=None):
    text = load_cells_text(plan, plan_file)
    if not str(text).strip(): cli_error("Pass a JSON batch edit plan or --plan_file")
    try: data = json.loads(text)
    except json.JSONDecodeError as exc: cli_error(f"Batch edit plan must be JSON: {exc}")
    if isinstance(data, list): data = {"operations": data}
    if not isinstance(data, dict) or not isinstance(data.get("operations"), list):
        cli_error("Batch edit plan must be a JSON object with an operations list")
    return data

In [ ]:
#| export
def _op_path(op, default_path=None):
    path = op.get("path") or default_path
    if not path: cli_error(f"Batch operation missing path: {op}")
    return Path(path)

In [ ]:
#| export
def _op_source(op):
    for key in ("source", "new", "text"):
        if key in op: return str(op[key])
    cli_error(f"Batch operation missing source/new/text: {op}")

In [ ]:
#| export
def _op_cells(op, default_cell_type="code"):
    if "cells" in op: return parse_cells(str(op["cells"]), op.get("cell_type", default_cell_type))
    return [parse_one_cell(_op_source(op), op.get("cell_type", default_cell_type))]

In [ ]:
#| export
def _op_diff(before, after, limit=24):
    lines = list(difflib.unified_diff(
        before.splitlines(), after.splitlines(), fromfile="before", tofile="after", lineterm="", n=2,
    ))
    if len(lines) > limit: lines = [*lines[:limit], "... diff truncated ..."]
    return "\n".join(lines)

In [ ]:
#| export
def _cell_source_hash(cell):
    return source_hash(cell_source(cell))


def _batch_detail(op, path, cell_id="", before="", after=""):
    detail = {
        "op": op.get("op"),
        "path": str(path),
        "cell_id": cell_id,
        "before_hash": source_hash(before) if before else "",
        "after_hash": source_hash(after) if after else "",
        "status": "planned",
        "diff": _op_diff(before, after) if before != after else "",
    }
    return detail

In [ ]:
#| export
def _apply_batch_op(nb, path, op, validate_code=True, default_cell_type="code"):
    kind = op.get("op")
    if kind in {"set_cell_source", "set_cell"}:
        _, cell = find_cell_by_id(nb.cells, op.get("cell_id"))
        before = cell_source(cell)
        source = _op_source(op)
        cell_type = op.get("cell_type")
        if cell_type: cell.cell_type = cell_type
        if validate_code and getattr(cell, "cell_type", None) == "code": validate_code_cells([mk_cell(source)])
        cell.source = source
        clear_outputs(cell)
        return [_batch_detail(op, path, cell.id, before, source)]
    if kind in {"insert_after_id", "insert_before_id"}:
        idx, anchor = find_cell_by_id(nb.cells, op.get("cell_id") or op.get("after_id") or op.get("before_id"))
        new_cells = _op_cells(op, default_cell_type=default_cell_type)
        if validate_code: validate_code_cells(new_cells)
        target = idx + 1 if kind == "insert_after_id" else idx
        inserted = []
        for offset, cell in enumerate(new_cells):
            clear_outputs(cell)
            nb.cells.insert(target + offset, cell)
            inserted.append({"cell_id": getattr(cell, "id", ""), "after_hash": _cell_source_hash(cell)})
        where = "after" if kind == "insert_after_id" else "before"
        return [{
            "op": kind,
            "path": str(path),
            "cell_id": getattr(anchor, "id", ""),
            "inserted": inserted,
            "status": "planned",
            "diff": f"inserted {len(new_cells)} cell(s) {where} id={getattr(anchor, 'id', '')}",
        }]
    if kind == "delete_cell_id":
        idx, cell = find_cell_by_id(nb.cells, op.get("cell_id"))
        before = cell_source(cell)
        del nb.cells[idx]
        return [_batch_detail(op, path, cell.id, before, "")]
    if kind == "replace_text":
        old = op.get("old_str", op.get("old"))
        new = op.get("new_str", op.get("new"))
        if old in {None, ""}: cli_error(f"replace_text needs old/old_str: {op}")
        if new is None: cli_error(f"replace_text needs new/new_str: {op}")
        _, matches, details = _replace_literal_in_notebook(nb, str(old), str(new), validate_code=validate_code, collect_details=True)
        if not matches: cli_error(f"No matches for {old!r} in {path}")
        return [
            {
                "op": kind,
                "path": str(path),
                "cell_id": item["cell_id"],
                "before_hash": item.get("before_hash", ""),
                "after_hash": item.get("after_hash", ""),
                "status": "planned",
                "diff": item["diff"],
            }
            for item in details
        ]
    cli_error(f"Unknown batch operation {kind!r}")

In [ ]:
#| export
def _detail_cell_matches(nb, detail):
    try:
        _, cell = find_cell_by_id(nb.cells, detail.get("cell_id"))
    except ValueError:
        return False
    return not detail.get("after_hash") or _cell_source_hash(cell) == detail.get("after_hash")


In [ ]:
#| export
def _detail_insert_matches(nb, detail):
    for item in detail.get("inserted", []):
        try:
            _, cell = find_cell_by_id(nb.cells, item.get("cell_id"))
        except ValueError:
            return False
        if _cell_source_hash(cell) != item.get("after_hash"): return False
    return True


In [ ]:
#| export
def _verify_batch_details(details):
    by_path = {}
    for detail in details:
        by_path.setdefault(Path(detail["path"]), []).append(detail)
    failed = []
    for path, path_details in by_path.items():
        nb = read_nb(path)
        for detail in path_details:
            op = detail.get("op")
            ok = True
            if op == "delete_cell_id":
                try:
                    find_cell_by_id(nb.cells, detail.get("cell_id"))
                    ok = False
                except ValueError:
                    ok = True
            elif op in {"insert_after_id", "insert_before_id"}:
                ok = _detail_insert_matches(nb, detail)
            else:
                ok = _detail_cell_matches(nb, detail)
            if not ok:
                failed_detail = dict(detail)
                failed_detail["status"] = "failed"
                failed.append(failed_detail)
    return failed


In [ ]:
#| export
def _format_batch_details(details):
    lines = []
    for item in details:
        index = f"#{item['index']} " if "index" in item else ""
        cell = f" id={item['cell_id']}" if item.get("cell_id") else ""
        status = f" [{item['status']}]" if item.get("status") else ""
        hashes = ""
        if item.get("before_hash") or item.get("after_hash"):
            hashes = f" {item.get('before_hash', '')}->{item.get('after_hash', '')}"
        lines.append(f"- {index}{item['op']}: {item['path']}{cell}{status}{hashes}")
        for inserted in item.get("inserted", []):
            lines.append(f"    inserted id={inserted.get('cell_id', '')} hash={inserted.get('after_hash', '')}")
        if item.get("diff"):
            lines.extend(f"    {line}" for line in item["diff"].splitlines())
    return "\n".join(lines)


### Verified batch edits

Batch plans are useful when an agent already knows the exact notebook edits it wants to make. They are also risky if the tool only says "applied" without checking disk. `batch_edit_nb` now writes the notebooks, reads them back, and reports a verification status plus before/after hashes for each operation.

In [ ]:
#| export
def batch_edit_nb(
    plan: Param("JSON edit plan, or - to read stdin", str, opt=False, nargs="?") = "",
    plan_file: str | None = None,  # Read the JSON plan from a UTF-8 file
    path: str | None = None,  # Default notebook path for operations that omit path
    dry_run: bool = True,  # Show the plan and diffs without writing
    validate_code: bool = True,  # Validate changed Python code before writing
    default_cell_type: str = "code",  # Default cell type for inserted cells without %% markers
):
    "Apply a JSON batch edit plan to one or more notebooks with locks, diffs, and read-back verification."
    data = _load_batch_plan(plan, plan_file)
    ops = data["operations"]
    paths = sorted({_op_path(op, path) for op in ops}, key=str)
    details = []
    exported = False
    with notebook_locks(*paths):
        notebooks = {nb_path: read_nb(nb_path) for nb_path in paths}
        for index, op in enumerate(ops, start=1):
            nb_path = _op_path(op, path)
            op_details = _apply_batch_op(
                notebooks[nb_path], nb_path, op,
                validate_code=validate_code, default_cell_type=default_cell_type,
            )
            for detail in op_details:
                detail["index"] = index
            details.extend(op_details)
        if not dry_run:
            for nb_path, nb in notebooks.items():
                stamp_notebook_metadata(nb)
                _write_nb(nb, nb_path)
                exported = export_notebook(nb, nb_path) is not None or exported
            failed = _verify_batch_details(details)
            if failed:
                for item in failed:
                    for detail in details:
                        if detail.get("index") == item.get("index") and detail.get("cell_id") == item.get("cell_id"):
                            detail["status"] = "failed"
                cli_error("Batch edit verification failed after writing:\n" + _format_batch_details(failed))
            for detail in details:
                detail["status"] = "verified"
    prefix = "Dry run: would apply" if dry_run else "Applied"
    msg = f"{prefix} {len(ops)} batch operations across {len(paths)} notebook(s)"
    if exported: msg += " and exported with nbdev"
    if details: msg += "\n" + _format_batch_details(details)
    print(msg)
    return cli_return({"paths": [str(path) for path in paths], "details": details})

### Splitting one chapter

`split_nb_chapter` moves one `##` chapter into a new nbdev notebook. It copies imports used by the moved code, imports source-notebook definitions that the moved chapter still references, and promotes referenced private source helpers by dropping the leading underscore. It is intentionally CLI-only, because splitting modules is a broad refactor that should be run deliberately from the shell.

In [ ]:
#| export
def _cell_lines(cell):
    source = cell_source(cell)
    return source.splitlines()


In [ ]:
#| export
def _default_exp_from_cells(cells):
    for cell in cells:
        for line in _cell_lines(cell):
            match = re.match(r"^\s*#\|\s*default_exp\s+(.+?)\s*$", line)
            if match: return match.group(1).strip()
    return None


In [ ]:
#| export
def _default_exp_for_dest(dest):
    dest = Path(dest)
    try:
        from nbdev.config import get_config
        cfg = get_config()
        nbs_path = Path(cfg.config_path) / cfg.nbs_path
        rel = dest.resolve().relative_to(nbs_path.resolve()).with_suffix("")
        return ".".join(rel.parts)
    except Exception:
        return dest.stem


In [ ]:
#| export
def _module_for_default_exp(default_exp, path=None):
    if not default_exp: return None
    try:
        from nbdev.config import get_config
        cfg = get_config(Path(path).parent if path else None)
        lib_name = str(cfg.lib_name)
    except Exception:
        lib_name = None
    if lib_name and not str(default_exp).startswith(f"{lib_name}."):
        return f"{lib_name}.{default_exp}"
    return str(default_exp)


In [ ]:
#| export
def _is_default_exp_cell(cell):
    return any(re.match(r"^\s*#\|\s*default_exp\s+", line) for line in _cell_lines(cell))


In [ ]:
#| export
def _code_ast(cell):
    if getattr(cell, "cell_type", None) != "code": return None
    try: return ast.parse(cell_source(cell))
    except SyntaxError: return None


In [ ]:
#| export
def _import_bound_names(node):
    if isinstance(node, ast.Import):
        return [alias.asname or alias.name.split(".", 1)[0] for alias in node.names]
    if isinstance(node, ast.ImportFrom):
        return [alias.asname or alias.name for alias in node.names if alias.name != "*"]
    return []


In [ ]:
#| export
def _node_source_from_cell(cell, node):
    lines = cell_source(cell).splitlines()
    start = min([node.lineno, *[d.lineno for d in getattr(node, "decorator_list", [])]]) - 1
    return "\n".join(lines[start:node.end_lineno]).strip("\n")


In [ ]:
#| export
def _definition_names(cells):
    names = {}
    for idx, cell in enumerate(cells):
        tree = _code_ast(cell)
        if tree is None: continue
        for node in tree.body:
            if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)):
                names[node.name] = idx
    return names


In [ ]:
#| export
def _bound_names(cells):
    names = set()
    for cell in cells:
        tree = _code_ast(cell)
        if tree is None: continue
        for node in tree.body:
            names.update(_import_bound_names(node))
            if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)):
                names.add(node.name)
        for node in ast.walk(tree):
            if isinstance(node, ast.Name) and isinstance(node.ctx, ast.Store): names.add(node.id)
    return names


In [ ]:
#| export
def _loaded_names(cells):
    names = set()
    for cell in cells:
        tree = _code_ast(cell)
        if tree is None: continue
        for node in ast.walk(tree):
            if isinstance(node, ast.Name) and isinstance(node.ctx, ast.Load): names.add(node.id)
    return names - _bound_names(cells) - set(dir(builtins))


In [ ]:
#| export
def _import_lines_for_names(cells, names):
    lines, seen = [], set()
    for cell in cells:
        tree = _code_ast(cell)
        if tree is None: continue
        for node in tree.body:
            bound = set(_import_bound_names(node))
            if not bound or not (bound & names): continue
            line = ast.unparse(node)
            if line not in seen:
                seen.add(line)
                lines.append(line)
    return lines


In [ ]:
#| export
def _symbol_replacements(promotions):
    return {old: new for old, new in promotions}


In [ ]:
#| export
def _replace_symbol_refs(source, promotions):
    for old, new in _symbol_replacements(promotions).items():
        source = re.sub(rf"(?<![\w.]){re.escape(old)}(?![\w])", new, source)
    return source


In [ ]:
#| export
def _apply_promotions(cells, promotions):
    if not promotions: return
    for cell in cells:
        if getattr(cell, "cell_type", None) == "code":
            cell.source = _replace_symbol_refs(cell_source(cell), promotions)
            clear_outputs(cell)


In [ ]:
#| export
def _insert_import_cell(cells, lines):
    lines = [line for line in lines if line]
    if not lines: return
    source = "#| export\n" + "\n".join(dict.fromkeys(lines))
    insert_at = 1 if cells and _is_default_exp_cell(cells[0]) else 0
    cells.insert(insert_at, mk_cell(source, cell_type="code"))


In [ ]:
#| export
def _split_chapter_plan(nb, chapter, dest, default_exp=None, promote_private=True):
    span = one_chapter(nb.cells, chapter)
    moved = [copy.deepcopy(cell) for cell in nb.cells[span["start"]:span["end"]]]
    remaining = [copy.deepcopy(cell) for idx, cell in enumerate(nb.cells) if not (span["start"] <= idx < span["end"])]
    moved_uses = _loaded_names(moved)
    remaining_uses = _loaded_names(remaining)
    outside_defs = _definition_names(remaining)
    moved_defs = _definition_names(moved)
    source_default_exp = _default_exp_from_cells(nb.cells)
    source_module = _module_for_default_exp(source_default_exp, path=dest)
    dest_default_exp = default_exp or _default_exp_for_dest(dest)
    dest_module = _module_for_default_exp(dest_default_exp, path=dest)

    source_deps = sorted(moved_uses & set(outside_defs))
    moved_deps = sorted(remaining_uses & set(moved_defs))
    promotions = []
    for name in source_deps:
        if name.startswith("_") and promote_private:
            public = name.lstrip("_")
            if public in outside_defs or public in moved_uses:
                cli_error(f"Cannot promote {name!r}: {public!r} already exists or is referenced")
            promotions.append((name, public))
    _apply_promotions(remaining, promotions)

    import_lines = _import_lines_for_names(remaining, moved_uses - set(outside_defs))
    if source_deps:
        if not source_module: cli_error("Source notebook needs #| default_exp before split dependencies can be imported")
        for name in source_deps:
            promoted = dict(promotions).get(name, name)
            import_lines.append(f"from {source_module} import {promoted} as {name}" if promoted != name else f"from {source_module} import {name}")
    dest_cells = [mk_cell(f"#| default_exp {dest_default_exp}", cell_type="code")]
    _insert_import_cell(dest_cells, import_lines)
    dest_cells.extend(moved)

    source_imports = []
    if moved_deps:
        if not dest_module: cli_error("Destination notebook needs #| default_exp before source dependencies can be imported")
        for name in moved_deps:
            source_imports.append(f"from {dest_module} import {name}")
    _insert_import_cell(remaining, source_imports)

    return {
        "span": span,
        "source_default_exp": source_default_exp,
        "dest_default_exp": dest_default_exp,
        "source_module": source_module,
        "dest_module": dest_module,
        "source_dependencies": source_deps,
        "moved_dependencies": moved_deps,
        "copied_imports": import_lines,
        "source_imports": source_imports,
        "promotions": promotions,
        "source_cells": remaining,
        "dest_cells": dest_cells,
    }


In [ ]:
#| export
def _format_split_plan(path, dest, chapter, plan, dry_run=False):
    prefix = "Dry run: would split" if dry_run else "Split"
    lines = [f"{prefix} chapter {chapter!r} from {path} -> {dest}"]
    lines.append(f"moved cells={len(plan['dest_cells']) - 1}; destination default_exp={plan['dest_default_exp']}")
    if plan["copied_imports"]:
        lines.append("destination imports:")
        lines.extend(f"- {line}" for line in plan["copied_imports"])
    if plan["source_imports"]:
        lines.append("source imports:")
        lines.extend(f"- {line}" for line in plan["source_imports"])
    if plan["promotions"]:
        lines.append("promoted source helpers:")
        lines.extend(f"- {old} -> {new}" for old, new in plan["promotions"])
    return "\n".join(lines)


In [ ]:
#| export
def split_nb_chapter(
    path: str,  # Source notebook path
    chapter: str,  # Chapter title string or regex to split out
    dest: str,  # Destination notebook path
    default_exp: str | None = None,  # Destination nbdev default_exp; defaults from dest path
    dry_run: bool = True,  # Show the split plan without writing notebooks
    force: bool = False,  # Overwrite dest if it already exists
    promote_private: bool = True,  # Promote referenced private source helpers by dropping the leading underscore
):
    "Split one ## chapter into a new nbdev notebook."
    path, dest = Path(path), Path(dest)
    if dest.exists() and not force: cli_error(f"Destination exists: {dest}; pass --force to overwrite")
    with notebook_locks(path, dest):
        nb = read_nb(path)
        plan = _split_chapter_plan(nb, chapter=chapter, dest=dest, default_exp=default_exp, promote_private=promote_private)
        msg = _format_split_plan(path, dest, chapter, plan, dry_run=dry_run)
        print(msg)
        if dry_run: return cli_return(plan)
        source_nb = new_nb(plan["source_cells"])
        dest_nb = new_nb(plan["dest_cells"])
        validate_code_cells([cell for cell in source_nb.cells if getattr(cell, "cell_type", None) == "code"])
        validate_code_cells([cell for cell in dest_nb.cells if getattr(cell, "cell_type", None) == "code"])
        stamp_notebook_metadata(source_nb)
        stamp_notebook_metadata(dest_nb)
        dest.parent.mkdir(parents=True, exist_ok=True)
        _write_nb(source_nb, path)
        _write_nb(dest_nb, dest)
        export_notebook(source_nb, path)
        export_notebook(dest_nb, dest)
    return cli_return(plan)


In [ ]:
plan_path = demo_path("batch_edit_plan.json")
try:
    with write_demo_notebook("batch_edit_target.ipynb") as nb_path:
        write_nb(str(nb_path), "%%code\nvalue = 1", replace=True)
        nb = read_nb(nb_path)
        cell = nb.cells[0]
        plan = {
            "operations": [
                {"op": "set_cell_source", "path": str(nb_path), "cell_id": cell.id, "source": "value = 2"},
                {"op": "insert_after_id", "path": str(nb_path), "cell_id": cell.id, "cells": "%%code\nassert value == 2"},
            ]
        }
        plan_path.write_text(json.dumps(plan), encoding="utf-8")
        batch_edit_nb(plan_file=str(plan_path), dry_run=True)
        out = StringIO()
        with redirect_stdout(out):
            batch_edit_nb(plan_file=str(plan_path), dry_run=False)
        applied = out.getvalue()
        print(applied, end="")
        nb = read_nb(nb_path)
        assert cell_source(nb.cells[0]) == "value = 2"
        assert "assert value == 2" in cell_source(nb.cells[1])
        assert "[verified]" in applied
        assert "->" in applied and "inserted id=" in applied
        print(f"verified {applied.count('[verified]')} batch edit records")
finally:
    remove_demo_path(plan_path)

Wrote 1 cells to nbs/data/batch_edit_target.ipynb using replace
Dry run: would apply 2 batch operations across 1 notebook(s)
- #1 set_cell_source: nbs/data/batch_edit_target.ipynb id=13f4ef6d [planned] e143e5f6659b->9cfe9326a093
    --- before
    +++ after
    @@ -1 +1 @@
    -value = 1
    +value = 2
- #2 insert_after_id: nbs/data/batch_edit_target.ipynb id=13f4ef6d [planned]
    inserted id=f631de99 hash=7e8872481e92
    inserted 1 cell(s) after id=13f4ef6d
Applied 2 batch operations across 1 notebook(s)
- #1 set_cell_source: nbs/data/batch_edit_target.ipynb id=13f4ef6d [verified] e143e5f6659b->9cfe9326a093
    --- before
    +++ after
    @@ -1 +1 @@
    -value = 1
    +value = 2
- #2 insert_after_id: nbs/data/batch_edit_target.ipynb id=13f4ef6d [verified]
    inserted id=74af8946 hash=7e8872481e92
    inserted 1 cell(s) after id=13f4ef6d
verified 2 batch edit records
